# Parameter Golf — Colab Runner

Free GPU validation for parameter-golf experiments.  
Edit `train_gpt.py` locally, commit, push, then hit **Run All** here.

**Runtime**: Go to `Runtime > Change runtime type > T4 GPU` (or A100 if available)

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────
!pip install -q sentencepiece numpy torch huggingface-hub datasets tqdm
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ── 2. Clone repo ────────────────────────────────────────────────────────
BRANCH = "autoresearch-findings"

!rm -rf parameter-golf
!git clone -b {BRANCH} https://github.com/lhubbard011/parameter-golf.git
%cd parameter-golf
!git log --oneline -3

In [ ]:
# ── 3. Download data (one-time, ~2 min) ──────────────────────────────────
import os
if not os.path.exists('data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin'):
    !python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 3
else:
    print('Data already downloaded')
!ls data/datasets/fineweb10B_sp1024/*.bin | wc -l

In [ ]:
# ── 4. Train! ─────────────────────────────────────────────────────────────
# Adjust MAX_WALLCLOCK_SECONDS for your GPU:
#   T4:  600s (10 min) — fewer steps but validates direction
#   A100: 300s (5 min) — comparable to our Lambda results

WALLCLOCK = 600  # seconds
DESC = "colab baseline"  # change this for each experiment

import time
t0 = time.time()

os.environ['MAX_WALLCLOCK_SECONDS'] = str(WALLCLOCK)
os.environ['ITERATIONS'] = '50000'
os.environ['VAL_LOSS_EVERY'] = '0'
os.environ['RUN_ID'] = DESC.replace(' ', '_')

!python3 -u train_gpt.py 2>&1 | tee run.log

elapsed = time.time() - t0
print(f'\nTotal time: {elapsed:.0f}s')

In [ ]:
# ── 5. Results ────────────────────────────────────────────────────────────
import re

with open('run.log') as f:
    log = f.read()

def extract(pattern):
    m = re.findall(pattern, log)
    return m[-1] if m else '?'

val_bpb = extract(r'val_bpb:([0-9.]+)')
val_loss = extract(r'val_loss:([0-9.]+)')
params = extract(r'model_params:([0-9]+)')
steps = extract(r'step:([0-9]+)/50000')
artifact = extract(r'int8\+zlib: ([0-9]+) bytes')

params_m = f'{int(params)/1e6:.1f}M' if params != '?' else '?'
artifact_mb = f'{int(artifact)/1048576:.1f}MB' if artifact != '?' else '?'

print('┌─────────────────────────────────────────┐')
print(f'│  {DESC:<39} │')
print('├─────────────────────────────────────────┤')
print(f'│  val_bpb:    {val_bpb:<27} │')
print(f'│  val_loss:   {val_loss:<27} │')
print(f'│  params:     {params_m:<27} │')
print(f'│  steps:      {steps:<27} │')
print(f'│  artifact:   {artifact_mb:<27} │')
print('└─────────────────────────────────────────┘')

In [ ]:
# ── 6. Download model ─────────────────────────────────────────────────────
from google.colab import files

model_name = f'{DESC.replace(" ", "_")}.int8.ptz'
!cp final_model.int8.ptz {model_name}
print(f'Model: {model_name} ({os.path.getsize(model_name)/1048576:.1f}MB)')
files.download(model_name)

In [ ]:
# ── 7. Quick inference test ───────────────────────────────────────────────
import sentencepiece as spm
import torch, io, zlib
import torch.nn.functional as F

# Load model using inference.py
exec(open('inference.py').read().split('def main')[0])  # load classes only

model = load_model('final_model.int8.ptz')
tokenizer = spm.SentencePieceProcessor(model_file='data/tokenizers/fineweb_1024_bpe.model')

prompt = "The most important thing about"
print(f'Prompt: {prompt}')
print('Output: ', end='')
generate(model, tokenizer, prompt, max_tokens=100)

---
## Running a different experiment

1. Edit `train_gpt.py` in your local repo
2. `git commit && git push`
3. Re-run cell 2 (clone) and cell 4 (train)

Or edit directly in Colab:
1. Click the file browser (left sidebar)
2. Open `parameter-golf/train_gpt.py`
3. Make changes
4. Re-run cell 4